## Imports

In [ ]:
%pip install pandas
%pip install scikit-learn
%pip install tensorflow
%pip install matplotlib
%pip install imbalanced-learn
%env SCIPY_ARRAY_API=1
%pip install xgboost lightgbm numpy
%pip install -U numpy scikit-learn
%pip install -U scikit-learn imbalanced-learn

In [ ]:


# 📦 Imports
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.utils.data_utils import load_and_preprocess_data
from src.utils.data_utils import preprocess_input
from src.model import build_weighted_model  # make sure this is imported
import csv
import pandas as pd
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import load_model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import recall_score
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTE
import joblib
import time


In [ ]:
# 1. Load model and scaler

model = load_model("../models/cnn_rnn_hybridensemble.keras")
scaler = joblib.load("../models/scaler_hybridensemble.pkl")

In [ ]:
# 2. Load your CSV
# df = pd.read_csv("../data/robot_sensor_data.csv")
df = pd.read_csv("../data/dataset.csv")

In [ ]:
# 3. Configure alert threshold
ALERT_THRESHOLD = 0.6   # Set as needed (e.g., 0.5, 0.7, 0.8...)

In [ ]:
# 4. Loop through each row, simulating real-time arrival
results = []  # Initialize results list




# for i, row in df.iterrows():
for i, row in df.head(52).iterrows():
    single_row_df = pd.DataFrame([row])
    X_row, _ = preprocess_input(single_row_df, scaler)
    prob = model.predict(X_row)[0][0]
    label = "FAILURE" if prob > 0.5 else "OK"
    
    # implement unsure label
    if prob > 0.4 and prob < 0.6:
        label = "UNSURE"
    
    # Save timestamp if exists, otherwise just use row index
    timestamp = row['date'] if 'date' in row else i
    
    # Log result to results list
    results.append({
        "timestamp": timestamp,
        "probability": prob,
        "label": label
    })
    
    if prob > ALERT_THRESHOLD:
        print(f"🚨 ALERT! Row {i}: FAILURE detected (prob: {prob:.2f}) 🚨")
    else:
        print(f"Row {i}: {label} (prob: {prob:.2f})")
    
    time.sleep(0.2)

# After the loop, export results
results_df = pd.DataFrame(results)
os.makedirs("results", exist_ok=True)
results_df.to_csv("../results/realtime_predictions_log.csv", index=False)
print("✅ Real-time predictions (with timestamps/row numbers) saved.")